# Three channel data creation

> Creating a three channel dataset for better training

In [ ]:
#| default_exp three_channel_with_diff.preprocessing.save_image

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2


In [ ]:
#| hide
from nbdev.showdoc import *
from pathlib import Path
from fastcore.all import *
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [ ]:
#| export
import torch

In [ ]:
#| export
from cv_tools.core import *

In [ ]:
#| export
def foo(): pass

In [ ]:
DATA_PATH = Path(r'/home/hasan/workspace/projects/data/ET')
im1_path = Path(DATA_PATH, 'main_im1_cropped')
im2_path = Path(DATA_PATH, 'main_im2_cropped')

DATA_PATH.ls(),im1_path.ls(),im2_path.ls()

In [ ]:
im_pairs = [(Path(i).name, Path(i).name.replace('image2', 'image1')) for i in im2_path.ls()]

In [ ]:
im1_path.ls(), im2_path.ls()

In [ ]:
im2_name, im1_name = im_pairs[0]
im1_name, im2_name

In [ ]:
im1_img = cv2.imread(Path(im1_path, im1_name).as_posix(), cv2.IMREAD_GRAYSCALE)
im2_img = cv2.imread(Path(im2_path, im2_name).as_posix(), cv2.IMREAD_GRAYSCALE)
im1_img.shape, im2_img.shape
im1_img_ = im1_img.astype(np.float32)/255.0
im2_img_ = im2_img.astype(np.float32)/255.0
im1_img_.shape, im2_img_.shape
diff_ch = (im1_img_ - im2_img_ + 1.0)/2.0
thr_ch = np.stack([im1_img_, im2_img_, diff_ch], axis=-1)

plt.imshow(thr_ch)












In [ ]:
#| export
def build_three_channel_image(im1_path, im2_path):
    im1_img = cv2.imread(Path(im1_path, im1_name).as_posix(), cv2.IMREAD_GRAYSCALE)
    im2_img = cv2.imread(Path(im2_path, im2_name).as_posix(), cv2.IMREAD_GRAYSCALE)
    im1_img_ = im1_img.astype(np.float32)/255.0
    im2_img_ = im2_img.astype(np.float32)/255.0
    diff_ch = (im1_img_ - im2_img_ + 1.0)/2.0
    thr_ch = np.stack([im1_img_, im2_img_, diff_ch], axis=-1)
    return thr_ch

In [ ]:
#| export
from itertools import product

In [ ]:
H,W,c = thr_ch.shape
H,W,c

In [ ]:
PATCH_SIZE = 400
STRIDE = 128
all_patches = []
for r, c in product(
    range(0, H, STRIDE),
    range(0, W, STRIDE),
    ):
    patch = np.zeros((PATCH_SIZE, PATCH_SIZE, thr_ch.shape[-1]), dtype=thr_ch.dtype)
    r_end = min(r+PATCH_SIZE,H)
    c_end = min(c+PATCH_SIZE,W)
    patch[:r_end-r, :c_end-c, :] = thr_ch[r:r_end, c:c_end, :]
    all_patches.append(patch)

In [ ]:
from cv_tools.core import *

In [ ]:
#| export
def extract_patches(
    im, # Opencv image 
    patch_size:int, # Patch size
    stride:int, # Stride
    ):
    "Extract patches from an image"
    H,W,c = im.shape
    all_patches = []
    for r, c in product(
        range(0, H, stride),
        range(0, W, stride),
    ):
        patch = np.zeros((patch_size, patch_size, im.shape[-1]), dtype=im.dtype)
        r_end = min(r+patch_size,H)
        c_end = min(c+patch_size,W)
        patch[:r_end-r, :c_end-c, :] = im[r:r_end, c:c_end, :]
        all_patches.append(patch)
    return all_patches


In [ ]:
assert len(extract_patches(thr_ch, PATCH_SIZE, STRIDE)) == 117


In [ ]:
pin_counts_ = {
    '149':29,
    '152':34,
    '153':23,
    '150':22, 
    '68':33
    }

In [ ]:
from new_test.patching.edge_segmentation_model import *
from new_test.patching.inference_batch_image import *
from pathlib import Path

In [ ]:
model_fn = Path(r'/home/hasan/workspace/projects/data/ET/models/model_20260115_142602_lr_0.0002_epochs_200.pth_best_val_0.8401_epoch_129.pth')
model_fn = Path(r'/home/hasan/workspace/projects/data/ET/models/epoch_106_val_dice_0.9557_05_08_27.pt')

In [ ]:
two_img_model = load_whole_pipeline_model(
    model_path=model_fn,
    in_channels=1,
    out_channels=1,
    tile_size=256,
	pred_threshold=0.5,
	dilation_iteration_rev=2,
	dilation_iteration=2,
	map_location='cuda',
	dropout_p=0.1,
	convert_to_onnx=False,
	opset_version=12,
	input_shape=(1,2,1152,1632),
	edge_model=True,
	onnx_path=Path('ETWAR_PatchLoetPinv00.onnx')

)

In [ ]:
im1_img = cv2.imread(Path(im1_path, im1_name).as_posix(), cv2.IMREAD_GRAYSCALE)
im2_img = cv2.imread(Path(im2_path, im2_name).as_posix(), cv2.IMREAD_GRAYSCALE)
comb_img = process_image_im1_im2_j(im1_img, im2_img)
#show_(im1_img)

In [ ]:
with torch.no_grad():
	two_img_model_pred = two_img_model(comb_img)

In [ ]:
show_(two_img_model_pred[0,0,:,:])

In [ ]:
show_(overlay_mask_border_on_image_frm_img(
    im2_img,
    two_img_model_pred[0,0,:,:].cpu().numpy()*255
))

In [ ]:




import nbdev; nbdev.nbdev_export('000_three_channel_data_creation.ipynb')